# GloVe GPU training on Kaggle

This notebook is submitted from VS Code with the Kaggle CLI. Kaggle runs it remotely; there is no live Kaggle kernel connection in VS Code.


In [ ]:
from pathlib import Path
import subprocess
import sys

import torch

repo_url = "https://github.com/khizerdevexp-commits/deep-learning-papers-pytorch.git"
project_root = Path("/kaggle/working/deep-learning-papers-pytorch")
if not (project_root / "implementations/models/glove.py").is_file():
    subprocess.run(["git", "clone", repo_url, str(project_root)], check=True)

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle did not attach a GPU; enable GPU in kernel settings")

sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("GPU:", torch.cuda.get_device_name(0))

c:\Users\Dell\Desktop\practice\deep-learning-papers-pytorch\implementations\models\glove.py:76: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:855.)
  return torch.sparse_coo_tensor(


Forward output shape: (4,)
Initial batch loss: 0.0179
Training objectives: [0.0182, 0.0135]
Combined embedding shape: (5, 8)


## Load the sample corpus and train

Internet access is enabled in `kernel-metadata.json` so this notebook can download WikiText-2. Co-occurrence construction runs on CPU; GloVe parameter updates use the selected Kaggle GPU.


In [ ]:
from implementations.models.glove import GloveModel
from implementations.utils.glove_data import load_glove_data

print("GloVe model and data loader imports succeeded")

Project root: /content/deep-learning-papers-pytorch
GPU: Tesla T4


In [7]:
%pip install -q datasets

In [ ]:
# WikiText-2 raw: https://huggingface.co/datasets/Salesforce/wikitext
# Review the dataset card for its current license and citation requirements.
from datasets import load_dataset
from pathlib import Path
from implementations.models.glove import GloveModel
from implementations.utils.glove_data import load_glove_data

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
sample_lines = wiki["train"][:2000]["text"]
corpus_path = Path("/kaggle/working/wikitext2_train_sample.txt")
corpus_path.write_text("\n".join(sample_lines), encoding="utf-8")

# Co-occurrence construction runs on CPU; use a bounded sample for this first run.
train_loader, vocabulary = load_glove_data(
    corpus_path,
    window_size=5,
    batch_size=1024,
    min_count=2,
    max_vocab_size=10000,
    shuffle=False,
)
cooccurrence = train_loader.dataset.cooccurrence_matrix
print(f"Vocabulary: {len(vocabulary):,}; observed pairs: {cooccurrence._nnz():,}")

# fit() uses AdaGrad and moves pair IDs/counts to the model's GPU.
device = torch.device("cuda")
model = GloveModel(len(vocabulary), embedding_dim=100).to(device)
history = model.fit(
    cooccurrence,
    epochs=10,
    learning_rate=0.05,
    batch_size=1024,
)
print("Summed epoch objectives:", [round(value, 3) for value in history])

artifact = {
    "state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    },
    "vocabulary": vocabulary,
    "embeddings": model.get_embeddings().detach().cpu(),
}
results_dir = Path("/kaggle/working/results")
results_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, checkpoint_path)
print("Saved Kaggle output:", checkpoint_path)

Vocabulary: 6,709; observed pairs: 420,422
Summed epoch objectives: [15451.9847869873, 6353.9105968475, 4768.471145153, 3717.8605418205, 3001.7714185715, 2512.5643043518, 2160.9556438923, 1898.722751379, 1693.9343409538, 1529.4730615616]
Saved /content/glove_wikitext2_sample.pt


In [ ]:
# Optional: continue the current model weights for 25 additional epochs.
# fit() creates a fresh AdaGrad optimizer, so its accumulators restart here.
continuation_history = model.fit(
    cooccurrence,
    epochs=25,
    learning_rate=0.05,
    batch_size=1024,
)
history.extend(continuation_history)
print("Objectives for continuation:", [round(value, 3) for value in continuation_history])

# Refresh the Kaggle output checkpoint.
artifact = {
    "state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    },
    "vocabulary": vocabulary,
    "embeddings": model.get_embeddings().detach().cpu(),
}
results_dir = Path("/kaggle/working/results")
results_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = results_dir / "glove_wikitext2_sample.pt"
torch.save(artifact, checkpoint_path)
print("Updated Kaggle output:", checkpoint_path)

Objectives for continuation: [1172.289, 1351.372, 472.732, 279.938, 208.55, 172.588, 150.894, 135.762, 124.604, 115.963, 108.959, 103.071, 98.006, 93.596, 89.63, 86.097, 82.878, 79.961, 77.257, 74.729, 72.426, 70.214, 68.164, 66.246, 64.421]
All epoch objectives: [15451.985, 6353.911, 4768.471, 3717.861, 3001.771, 2512.564, 2160.956, 1898.723, 1693.934, 1529.473, 2731.653, 1776.208, 1062.46, 824.755, 700.355, 615.287, 552.291, 502.733, 461.508, 427.051, 1504.098, 1450.745, 617.565, 419.332, 338.586, 292.931, 262.371, 239.661, 221.96, 207.262, 194.725, 183.978, 174.433, 165.916, 158.294, 1172.289, 1351.372, 472.732, 279.938, 208.55, 172.588, 150.894, 135.762, 124.604, 115.963, 108.959, 103.071, 98.006, 93.596, 89.63, 86.097, 82.878, 79.961, 77.257, 74.729, 72.426, 70.214, 68.164, 66.246, 64.421]
Updated /content/glove_wikitext2_sample.pt


## Retrieve and inspect results

After the Kaggle run completes, open PowerShell in `C:\Users\Dell\Desktop\practice` and run:

```powershell
& '.\.venv\Scripts\kaggle.exe' kernels status khizercheema/glove-wikitext-2-gpu-training
& '.\.venv\Scripts\kaggle.exe' kernels output khizercheema/glove-wikitext-2-gpu-training -p '.\deep-learning-papers-pytorch' --file-pattern '.*glove_wikitext2_sample\.pt$'
```

The notebook writes the checkpoint to `/kaggle/working/results/glove_wikitext2_sample.pt`. The output command preserves the remote `results` folder, so the local file lands at `deep-learning-papers-pytorch\results\glove_wikitext2_sample.pt`. Do not use `-p '.\deep-learning-papers-pytorch\results'`, or the nested remote folder would create `results\results`.


In [ ]:
import torch
import torch.nn.functional as F

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
vocabulary = checkpoint["vocabulary"]
vectors = F.normalize(checkpoint["embeddings"], dim=1)
id_to_word = {index: word for word, index in vocabulary.items()}

def nearest_neighbors(word, count=10):
    if word not in vocabulary:
        print(f"{word!r} is not in the vocabulary")
        return
    word_id = vocabulary[word]
    scores = vectors @ vectors[word_id]
    scores[word_id] = -torch.inf
    for index in torch.topk(scores, min(count, len(vocabulary) - 1)).indices.tolist():
        print(id_to_word[index], round(scores[index].item(), 3))

nearest_neighbors("language")

wasp 0.523
pound 0.411
hodgkin 0.389
concert 0.385
carbenoid 0.376
eight 0.375
monte 0.371
themed 0.364
agrarian 0.357
anti 0.357


Training objective is not an accuracy score. Use the printed objective to check optimization, then evaluate semantic quality with a held-out word-similarity or analogy benchmark. The WikiText-2 sample is a pipeline-learning run, not a reproduction of the paper's reported results.


# Submit this notebook from VS Code

From the repository root, authenticate once with `kaggle auth login`, then push this staged notebook:

```powershell
& '..\.venv\Scripts\kaggle.exe' kernels push -p '.\kaggle\glove-wikitext2' --accelerator NvidiaTeslaT4
```

The kernel metadata enables internet for the Hugging Face dataset download and selects a GPU. Check the run status and retrieve the output using the commands in the results section above.
